In [1]:
import pandas as pd

In [2]:
vacancies_df = pd.read_csv('../data/processed/cleaned_vacancies.csv')

vacancies_df.head(1)

,vacancy_id,title,author_name,description,city,salary_min,salary_max,requirements,conditions,metro,currency,experience_min,experience_max,tags,remote_type,time_type,author_id
0,49313809,Golang Developer (Кипр),Space307,Мы в Space307 разрабатываем международную торг...,Санкт-Петербург,251322.0,NaN,"Программист, разработчик",Условия обсуждаются на собеседовании,NaN,RUB,3,6.0,"docker, golang, redis, английский язык, kafka",OFFICE,FULL,c266cc48-8be0-4b5a-8e4f-03b62a9a456c


In [3]:
vacancies_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 47325 entries, 0 to 47324
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   vacancy_id      47325 non-null  int64  
 1   title           47325 non-null  str    
 2   author_name     47325 non-null  str    
 3   description     47325 non-null  str    
 4   city            47325 non-null  str    
 5   salary_min      15079 non-null  float64
 6   salary_max      10037 non-null  float64
 7   requirements    47286 non-null  str    
 8   conditions      47325 non-null  str    
 9   metro           0 non-null      float64
 10  currency        47325 non-null  str    
 11  experience_min  47325 non-null  int64  
 12  experience_max  45759 non-null  float64
 13  tags            47325 non-null  str    
 14  remote_type     47325 non-null  str    
 15  time_type       47325 non-null  str    
 16  author_id       47325 non-null  str    
dtypes: float64(4), int64(2), str(11)
memory us

In [4]:
import pandas as pd

def _build_super_string(row: pd.Series) -> str:

    title = str(row['title']) if pd.notna(row['title']) else ''
    author = str(row['author_name']) if pd.notna(row['author_name']) else ''
    reqs = str(row['requirements']) if pd.notna(row['requirements']) else ''
    tags = str(row['tags']) if pd.notna(row['tags']) else ''
    
    return f"{title} {author} {reqs} {tags}".strip()


def check_recomendations(ids_to_check: list):

    target_id = ids_to_check[0]
    rec_ids = ids_to_check[1:]

    target_df = vacancies_df[vacancies_df['vacancy_id'] == target_id]
    
    recs_df = vacancies_df[vacancies_df['vacancy_id'].isin(rec_ids)].set_index('vacancy_id').loc[rec_ids].dropna(how='all').reset_index()

    print("ОРИГИНАЛЬНАЯ ВАКАНСИЯ")
    for index, row in target_df.iterrows():
        print(f"ID: {row['vacancy_id']} | {row['title']} | Компания: {row['author_name']}")
        print("-" * 60)
        
        super_string = _build_super_string(row)
        print(f"  Суперстрока (passage): {super_string}")
        
        print(' ', row['description'], "\n")


    print("РЕКОМЕНДАЦИИ")

    for index, row in recs_df.iterrows():
        print("=" * 80)
        print(f"ID: {row['vacancy_id']} | {row['title']} | Компания: {row['author_name']}")
        print("-" * 60)
        
        super_string = _build_super_string(row)
        print(f"  Суперстрока (passage): {super_string}")
        
        print(' ', row['description'], "\n")


def show_search_results(query: str, response_ids: list):

    recs_df = vacancies_df[vacancies_df['vacancy_id'].isin(response_ids)].set_index('vacancy_id').loc[response_ids].dropna(how='all').reset_index()

    print(f"--- Результаты текстового поиска по запросу: '{query}' ---")
    
    for index, row in recs_df.iterrows():
        print("=" * 80)
        print(f"ID: {row['vacancy_id']} | {row['title']} | Компания: {row['author_name']}")
        print("-" * 60)
        
        super_string = _build_super_string(row)
        print(f"  Суперстрока (passage): {super_string}")
        
        print(' ', row['description'], '\n')

In [5]:
%load_ext autoreload
%autoreload 2

import sys
import os

project_root = os.path.abspath('..')

if project_root not in sys.path:
    sys.path.append(project_root)

from src.vacancy_rec_sys import VacancyRecSys

d:\Projects\vacancy recommendations\.venv_win\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
rec_sys = VacancyRecSys()

rec_sys.initialize(initial_df=vacancies_df)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4605.01it/s]


Кэш не найден. Начинается сборка индекса...
Индекс успешно собран и сохранен на диск.


In [7]:
query = 'QA Automation Engineer'
response = rec_sys.search(query, limit=3, offset=0)
response_ids = [item.vacancy_id for item in response.items]

show_search_results(query, response_ids)


--- Результаты текстового поиска по запросу: 'QA Automation Engineer' ---
ID: 50033289 | QA Automation Engineer | Компания: Работа.ру
------------------------------------------------------------
  Суперстрока (passage): QA Automation Engineer Работа.ру Тестировщик php, api, qa, postman
  Работа.ру — один из главных job-сервисов России. Мы помогаем людям не только найти работу, но и прокачать свои навыки или полностью поменять сферу, еще до собеседования узнать, насколько компания надежная, какие в ней зарплаты, и много чего еще. А компаниям — находить правильных сотрудников, даже если нужно нанять 300 человек за месяц, вести воронку соискателей в удобной CRM и строить свой бренд работодателя. Мы ожидаем:  Знания об устройстве клиент-серверного взаимодействия; Представление об устройствах REST-архитектуры; Навыки работы с приложениями для сниффинга траффика; Навыки работы с POSTMAN; Способность самостоятельно локализировать ошибки; Понимание архитектуры работы тестируемого продукта; Спо

In [8]:
random_row = vacancies_df.sample(n=1)

target_id = random_row['vacancy_id'].iloc[0]

response_ids = [target_id]
response = rec_sys.get_recommendations(target_id, top_k=5)


for item in response.items:
    print(f"ID Вакансии: {item.vacancy_id}, Скор: {item.score:.4f}")
    response_ids.append(item.vacancy_id)

print()
check_recomendations(response_ids)

ID Вакансии: 49997914, Скор: 0.9581
ID Вакансии: 49838683, Скор: 0.9553
ID Вакансии: 49111236, Скор: 0.9538
ID Вакансии: 49693971, Скор: 0.9511
ID Вакансии: 49954611, Скор: 0.9499

ОРИГИНАЛЬНАЯ ВАКАНСИЯ
ID: 49467000 | HTML верстальщик | Компания: Кейклэбс
------------------------------------------------------------
  Суперстрока (passage): HTML верстальщик Кейклэбс Программист, разработчик javascript, sass, git, less, html5
  Приветствуем, ищем верстальщика или начинающего фронта. Офис на петроградской, компьютер iMac 27 + второй монитор 27. Гибка и дружелюбная компания. Главное чтобы ты уже умел хорошо верстать, а не был стажером.   Обязанности: разработка фронтенда Требования: уверенные знания верстки на flex и grid; знание SCSS. Адаптивная и кроссбраузерная верстка; less/sass; знание шаблонизатора Pug; знание JavaScript, webpack; умение работать с системами контроля версий (git); умение работать с анимацией будет плюсом (Canvas, WebGL) понимание структуры сайтов на БУС будет плюсом 

In [9]:
# тестровая вакансия
beekeeper_vacancy = {
    "vacancy_id": 9999999,
    "title": "Главный пасечник / Специалист по пчеловодству",
    "author_name": "ООО «Медовая Долина»",
    "requirements": "Опыт работы с пчелосемьями от 3 лет, знание методов профилактики болезней пчел, умение откачивать мед, отсутствие аллергии на укусы.",
    "tags": "пчеловодство пасека мед сельское-хозяйство агропром",
    "description": "Ищем заботливого и опытного пасечника для нашей фермы. В ведении будет более 200 ульев. Основные задачи: регулярный осмотр пчелосемей, расширение гнезд, откачка товарного меда, сбор прополиса и подготовка пчел к зимовке. Работа на свежем воздухе вдали от городской суеты."
}

rec_sys.add_vacancy(beekeeper_vacancy)
print(f"Вектор добавлен! Всего в базе: {rec_sys.index.ntotal}")

new_row_df = pd.DataFrame([beekeeper_vacancy])
vacancies_df = pd.concat([vacancies_df, new_row_df], ignore_index=True)


Вектор добавлен! Всего в базе: 47326


In [10]:
query = "уход за пчелами и сбор меда"
response = rec_sys.search(query, limit=3, offset=0)
response_ids = [item.vacancy_id for item in response.items]

show_search_results(query, response_ids)

--- Результаты текстового поиска по запросу: 'уход за пчелами и сбор меда' ---
ID: 9999999 | Главный пасечник / Специалист по пчеловодству | Компания: ООО «Медовая Долина»
------------------------------------------------------------
  Суперстрока (passage): Главный пасечник / Специалист по пчеловодству ООО «Медовая Долина» Опыт работы с пчелосемьями от 3 лет, знание методов профилактики болезней пчел, умение откачивать мед, отсутствие аллергии на укусы. пчеловодство пасека мед сельское-хозяйство агропром
  Ищем заботливого и опытного пасечника для нашей фермы. В ведении будет более 200 ульев. Основные задачи: регулярный осмотр пчелосемей, расширение гнезд, откачка товарного меда, сбор прополиса и подготовка пчел к зимовке. Работа на свежем воздухе вдали от городской суеты. 

ID: 49411837 | Мастер/ремонт/Специалист по обслуживанию майнинг ферм | Компания: ЮФ Модус Вита
------------------------------------------------------------
  Суперстрока (passage): Мастер/ремонт/Специалист по обслу

In [ ]:
# rec_sys.save_to_disk()